# 03 · SARIMA × SARIMAX — tipologias e modelo geral

Esta é a apresentação atual da etapa SARIMAX do TCC. A pergunta é se a inclusão de informações sobre desastres melhora o ajuste e a previsão **condicional** da variação da inadimplência, em relação a um SARIMA sem exógenas.

Há dois conjuntos de resultados, estimados sobre as mesmas séries e amostras: blocos climáticos de **0 a K** e blocos apenas de **3 a K**. Os segundos avaliam a sensibilidade à exclusão dos meses contemporâneo, t−1 e t−2. A versão antiga do caderno, com subconjuntos de tipologias e apenas blocos curtos, foi transferida para `historico/`.

Os resultados abaixo são lidos diretamente dos CSVs exportados pelas rotinas de estimação. Para reproduzir os ajustes, siga os comandos na última seção.

## 1. Séries e amostras

- Unidade: 27 UFs e o agregado Brasil, em dois recortes sobrepostos (pré-pandemia até jan/2020 e total até dez/2024).
- Resposta: `y_t = 100 Δlog(I_t)`, em que `I_t` é a taxa de inadimplência.
- Exposição: `x_j,t = Δlog(1 + D_j,t)` para cada uma das 16 tipologias do Atlas.
- O Brasil é obtido pela razão dos saldos agregados de crédito; não pela média das taxas estaduais.
- Todos os K usam a mesma amostra, descontados os 24 primeiros meses. Há 48 meses de treino e 12 de teste no pré, 95 de treino e 24 de teste no total.
- A ordem temporal do SARIMA é escolhida por BIC entre cinco especificações sazonais de periodicidade anual (12). Essa ordem é mantida no SARIMAX pareado.

O treino define as ordens, as defasagens e as tipologias estimáveis. O RMSE e o MAE são calculados somente no teste cronológico.

## 2. Seleção por tipologia e composição do modelo geral

Para cada UF, período e tipologia, os candidatos são blocos consecutivos de exposição:

| Referência | Sensibilidade |
| --- | --- |
| `[0]`, `[0,1]`, ..., `[0,...,24]` | `[3]`, `[3,4]`, ..., `[3,...,24]` |

O menor BIC **no treino** define K entre os candidatos que têm suporte, variação, posto completo, condicionamento aceitável e convergência. A grade de candidatos inclui bloqueios documentados e, portanto, seu tamanho não é a quantidade de ajustes bem-sucedidos.

O SARIMAX geral reúne as tipologias estimáveis e seus K escolhidos individualmente. Quando a matriz combinada não é identificável, a inclusão segue suporte e verificações de posto, sem usar valores-p do modelo geral. A saída registra as tipologias excluídas e seus motivos.

Não se estimam todas as 16 tipologias simultaneamente nos modelos individuais: são 16 modelos separados por UF/período, antes de construir um geral.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
BASE = ROOT / "outputs" / "tables"
PASTA_0 = BASE / "sarimax_por_tipologia_k0_24_20261026"
PASTA_3 = BASE / "sarimax_por_tipologia_k3_24_20261026"

def ler(pasta, nome):
    arquivo = pasta / f"{nome}.csv"
    if not arquivo.exists():
        raise FileNotFoundError(f"Tabela não encontrada: {arquivo}")
    return pd.read_csv(arquivo, encoding="utf-8-sig")

individual_0 = ler(PASTA_0, "resumo")
individual_3 = ler(PASTA_3, "resumo")
geral_0 = ler(PASTA_0, "geral")
geral_3 = ler(PASTA_3, "geral")
pares = ler(PASTA_3, "comparacao_individuais")
assert len(individual_0) == len(individual_3) == len(pares) == 896
assert len(geral_0) == len(geral_3) == 56
print("Tabelas carregadas: 896 aplicações individuais e 56 aplicações gerais por estratégia.")

## 3. Resultados dos SARIMAX individuais

A comparação com o SARIMA usa `ΔBIC = BIC(SARIMAX) − BIC(SARIMA)`; valores negativos favorecem a inclusão da exposição. Para o RMSE, a melhora percentual é positiva quando o erro de teste é menor.

| Critério | 0 a K | 3 a K |
| --- | ---: | ---: |
| Modelos individuais estimáveis | 452 | 452 |
| BIC menor que SARIMA | 22 | 29 |
| RMSE de teste menor que SARIMA | 167 | 179 |
| Ambos favoráveis | **4** | **5** |
| K no limite inferior | 420 em K=0 | 424 em K=3 |

Em 452 comparações pareadas, a janela 3 a K teve BIC menor que a janela 0 a K em 236 e RMSE menor em 236; em 91, ambos foram melhores. A diferença mediana entre estratégias foi pequena. A concentração no primeiro K permitido permanece nas duas grades e não demonstra um atraso climático de duração determinada.

In [ ]:
def indicadores(tab):
    est = tab.loc[tab["status"].eq("estimado")]
    return pd.Series({
        "Estimáveis": len(est),
        "BIC menor que SARIMA": int(est["delta_BIC"].lt(0).sum()),
        "RMSE menor que SARIMA": int(est["melhora_rmse_pct"].gt(0).sum()),
        "Ambos melhores": int((est["delta_BIC"].lt(0) & est["melhora_rmse_pct"].gt(0)).sum()),
    })

indicadores_0 = indicadores(individual_0)
indicadores_3 = indicadores(individual_3)
resumo = pd.concat([indicadores_0.rename("0 a K"), indicadores_3.rename("3 a K")], axis=1)
display(resumo)
assert indicadores_0["Ambos melhores"] == 4
assert indicadores_3["Ambos melhores"] == 5

est_3 = individual_3.loc[individual_3["status"].eq("estimado")]
distribuicao = est_3.groupby(["periodo", "K"]).size().unstack(fill_value=0)
display(distribuicao)

### Casos em que os dois critérios melhoraram

Apenas cinco modelos individuais com lags de 3 a K apresentaram simultaneamente BIC menor e RMSE de teste inferior ao SARIMA:

- **Pré:** MG/Chuvas intensas (K=3; ΔBIC −1,49; RMSE −5,56%), MS/Granizo (K=3; −4,26; −7,41%) e RN/Estiagem e seca (K=3; −2,96; −12,42%).
- **Total:** AP/Doenças infecciosas (K=3; −0,13; −0,05%) e RN/Estiagem e seca (K=3; −9,81; −0,86%).

São ganhos localizados. Parte das diferenças é pequena, especialmente no período completo. A tabela seguinte recupera os valores sem arredondamento da saída da execução.

In [ ]:
favoraveis = individual_3.loc[
    individual_3["status"].eq("estimado")
    & individual_3["delta_BIC"].lt(0)
    & individual_3["melhora_rmse_pct"].gt(0),
    ["periodo", "unidade", "tipologia", "K", "delta_BIC", "melhora_rmse_pct"],
].sort_values(["periodo", "unidade"])
display(favoraveis.round(3))
assert len(favoraveis) == 5

comparaveis = pares.loc[pares["ambos_estimaveis"]]
print("Comparações pareadas:", len(comparaveis))
print("3 a K com BIC menor:", int(comparaveis["BIC_melhor_3aK"].sum()))
print("3 a K com RMSE menor:", int(comparaveis["RMSE_melhor_3aK"].sum()))

## 4. Modelos gerais e agregado nacional

Foram estimados 55 gerais de 56 possíveis. O DF no pré não apresentou tipologia individual estimável. Na sensibilidade 3 a K, três gerais tiveram BIC menor que o SARIMA e nove tiveram RMSE menor; **nenhum** reuniu as duas melhorias. Os três gerais com BIC favorável (AC, PI e RR, todos pré) pioraram o RMSE em 40,42%, 145,22% e 186,54%.

No Brasil, todas as 16 tipologias puderam entrar no geral:

| Recorte | BIC do SARIMA | ΔBIC do SARIMAX 3–K | Melhora RMSE 3–K |
| --- | ---: | ---: | ---: |
| Pré | 214,69 | +51,04 | −36,87% |
| Total | 459,74 | +45,89 | −51,86% |

A inclusão das exposições **piorou o BIC e o erro de previsão condicional do Brasil** nos dois recortes.

In [ ]:
for nome, tabela in [("0 a K", geral_0), ("3 a K", geral_3)]:
    est = tabela.loc[tabela["status"].eq("estimado")]
    print(nome, "| gerais:", len(est),
          "| BIC favorável:", int(est["delta_BIC"].lt(0).sum()),
          "| RMSE favorável:", int(est["melhora_rmse_pct"].gt(0).sum()),
          "| ambos:", int((est["delta_BIC"].lt(0) & est["melhora_rmse_pct"].gt(0)).sum()))
    brasil = est.loc[est["unidade"].eq("Brasil"),
                     ["periodo", "n_tipologias_incluidas", "numero_regressoras",
                      "bic_sarima", "bic_sarimax", "delta_BIC", "melhora_rmse_pct"]]
    display(brasil.round(3))

## 5. Diagnóstico e interpretação

No SARIMAX 3 a K, o teste de Ljung–Box das inovações padronizadas rejeitou a hipótese de ausência de autocorrelação em 115 dos 452 modelos individuais. Os valores-p dos coeficientes são condicionais à escolha de ordens e defasagens; mesmo com ajuste BH exploratório, não devem ser tratados como inferência seletiva formal.

O teste é uma **previsão condicional de um passo**, com a observação de inadimplência até t−1 usada na atualização do estado. No protocolo 0 a K, a exposição do próprio mês t também está disponível, de modo que o resultado não é uma previsão produzida antes de observar os eventos de t. O protocolo 3 a K evita a exposição contemporânea, mas continua sendo uma avaliação condicional de um passo.

A medida de inadimplência acima de 90 dias é estoque. Não informa a data em que cada operação começou a atrasar. A escolha de K=3 pode resultar da penalização do BIC e do limite inferior da grade, e não comprova que o efeito dos desastres começa três meses depois. Nem a comparação preditiva nem os testes identificam efeito causal.

## 6. Como reproduzir as estimações

As tabelas já exportadas são consultadas pelas células acima. Para reestimar, prepare a base processada com o SHA-256 indicado no [README principal](../README.md) e execute, na raiz:

```bash
python src/executar_sarimax_tipologias_20261026.py --workers 4
python src/sarimax_por_tipologia_k0_24_20261026.py --fase gerais
python src/sarimax_por_tipologia_k0_24_20261026.py --fase consolidar

python src/executar_sarimax_tipologias_20261026.py --workers 4 --inicio-lags 3
python src/sarimax_por_tipologia_lags3_24_20261026.py --fase gerais
python src/sarimax_por_tipologia_lags3_24_20261026.py --fase consolidar
python src/comparar_sarimax_lags3_24_20261026.py
python src/verificar_sarimax_lags3_24_20261026.py
```

Detalhes dos candidatos bloqueados, critérios de estimabilidade e comparações: [protocolo 0–K](../docs/sarimax_por_tipologia_e_geral_20261026.md) e [sensibilidade 3–K](../docs/sarimax_sensibilidade_3a24_20261026.md).